#### > 활성함수 ( Activation Function )
: 신경망의 선형 연산 결과에 비선형성을 추가하여 복잡한 패턴을 학습할 수 있도록 돕는 함수

- 활성화 함수의 목적
    - 은닉층에 비선형성 추가
    - 출력값을 조절
    - 문제에 맞는 출력을 생성

#### 1. Sigmoid 함수
: 입력값 0 ~ 1사이의 값으로 변환하는 함수

- 특징
    - 출력 범위를 0과 1 사이로 제한
    - 입력값이 0이면 출력값은 0.5
    - 입력값이 커지면 출력은 1에 가까워짐
    - 입력값이 작아지면 출력은 0에 가까워짐
    - 이진분류에서 한 클래스에 속할 확률처럼 해석할 수 있는 값들을 만들 때 사용

- 단점
    - 입력값의 절대값이 아주 커지면 기울기가 작아지는 포화현상이 일어남
    - 은닉층에 어려번 사용하면 기울기 소실 문제가 생길 수 있음


In [1]:
import torch
import torch.nn as nn

In [ ]:
x = torch.tensor([-10., -2, -1, 0, 1, 2, 10])

output = torch.sigmoid(x)

print(output)

tensor([4.5398e-05, 1.1920e-01, 2.6894e-01, 5.0000e-01, 7.3106e-01, 8.8080e-01,
        9.9995e-01])


In [6]:
logits = torch.tensor([-2.0, -0.5, 0, 0.8, 2])  # 모델의 원래 출력값이지만 임의로 작성했음
print('모델 층을 통과한 출력(Logits) :', logits)

si = torch.sigmoid(logits)
print('sigmoid 출력:', si)

class0 = si >= 0.5
print('분류결과 :', class0)

class1 = (si >= 0.5).int()  
# .int() 는 bool값을 정수형으로 변환
#           False = 0
#           True = 1
print('분류결과 :', class1)

모델 층을 통과한 출력(Logits) : tensor([-2.0000, -0.5000,  0.0000,  0.8000,  2.0000])
sigmoid 출력: tensor([0.1192, 0.3775, 0.5000, 0.6900, 0.8808])
분류결과 : tensor([False, False,  True,  True,  True])
분류결과 : tensor([0, 0, 1, 1, 1], dtype=torch.int32)


#### 2. ReLU 함수
: 입력값이 0보다 작으면 무조건 0을 출력하고, 0보다 크면 그대로 출력하는 함수

- 특징
    - 계산이 단순하고 효율적임
    - 양수 영역에서는 기울기가 1이므로 기울기 소실이 적음
    - 일반적인 신경망의 은닉층에서 많이 사용됨

- 단점
    - 음수 영역에서는 출력과 기울기가 0임
    - 학습 일부 유런이 음수 영역에 머물면 업데이트가 되지 않음

In [7]:
# 표기법 1: nn.ReLU()
# 표기법 2: torch.relu()
x = torch.tensor([-5., -2, -1, 0, 1, 2, 5])
output = torch.relu(x)

print('입력:', x)
print('출력:', output)

입력: tensor([-5., -2., -1.,  0.,  1.,  2.,  5.])
출력: tensor([0., 0., 0., 0., 1., 2., 5.])


#### 3. Tanh 하이퍼볼릭 탄젠트
: 입력값이 -1 ~ 1 사이의 값으로 변환

- 특징
    - 입력값이 0이면 출력값도 0
    - 양수입력은 1에 가깝고, 음수입력은 -1에 가까움
    - 출력 중심이 0 <=> sigmoid = 0.5
- 단점
    - 입력값의 절대값이 매우 커지면 기울기가 작아지는 포화 현상 발생(sigmoid와 동일)
    - 여러번 사용해서 기울기가 계속 작아지면 소실되는 문제

In [8]:
x = torch.tensor([-10., -2, -1, 0, 1, 2, 10])

output = torch.tanh(x)

print('입력:', x)
print('출력:', output)

입력: tensor([-10.,  -2.,  -1.,   0.,   1.,   2.,  10.])
출력: tensor([-1.0000, -0.9640, -0.7616,  0.0000,  0.7616,  0.9640,  1.0000])


#### 4. Softmax 
: 입력받은 실수 값들을 0과 1사이의 값으로 변환하고, 이 값들의 총합이 정확히 1이 되도록 만들어 확률분포로 해석할 수 있게 해줌

- 특징
    - 서로 배타적인 여러 클래스중 하나를 선택하는 다중분류의 출력에 사용
    - 각 출력값은 0 ~ 1 사이의 값
    - 모든 클래스의 출력값을 더하면 1이 됨
    - 출력값이 각 클래스에 대한 예측 확률처럼 해석할 수 있음

- 옵션 dim
    - dim= 0 첫번째 차원을 기준으로 Softmax를 계산
    - dim= 1 두번째 차원을 기준으로 Softmax를 계산
    - batch 데이터가 (batch_size, num_classes) 형태라면 일반적으로 dim=1을 사용

- 클래스란?
    - 분류 문제의 목표값
    - 정수형으로 처리하고 1차원 형태로 구성

In [ ]:
logits = torch.tensor([2, 1, 0.1])  # logit이 작을수록(음수포함) 확률값이 줄어듦

sm = torch.softmax(logits, dim=0)

print('Logits : ', logits)
print('Softmax : ', sm)
print('확률의 합:', sm.sum())

# 가장 높은 확률을 가진 클래스를 선택
print('예측 클래스 인덱스:', torch.argmax(sm))

Logits :  tensor([2.0000, 1.0000, 0.1000])
Softmax :  tensor([0.6590, 0.2424, 0.0986])
확률의 합: tensor(1.0000)
예측 클래스 인덱스: tensor(0)


In [13]:
# 음수 포함 로짓
logits = torch.tensor([-2, -1, -0.1])

sm = torch.softmax(logits, dim=0)

print('Logits : ', logits)
print('Softmax : ', sm)
print('확률의 합:', sm.sum())

# 가장 높은 확률을 가진 클래스를 선택
print('예측 클래스 인덱스:', torch.argmax(sm))

Logits :  tensor([-2.0000, -1.0000, -0.1000])
Softmax :  tensor([0.0961, 0.2613, 0.6426])
확률의 합: tensor(1.0000)
예측 클래스 인덱스: tensor(2)


- 예를 들어 데이터 2개를 각각 3개 클래스로 분류한다면 shape(2,3)
    - 각 행(row)은 하나의 데이터에 대한 클래스의 합이 1이 되어야 함
    - 각 데이터별로 클래스 확률의 합이 1이 되어야 함.

- 비교
    - dim=0, 같은 클래스의 열을 기준으로 서로 다른 데이터 사이의 합이 1이됨
    - dim=1, 각 데이터(행)의 클래스 확률 합이 1이 됨

- 동물 사진 데이터를 입력했을 때 분류되는 클래스가 ( 개, 고양이, 토끼 ) 인 경우
    - 각 클래스로 분류될 확률 중 제일 큰 값에 대한 결과를 출력하도록 하는 함수
    - if torch.argmax(Softmax) = tensor[0], then '이 사진은 개 사진입니다.'

In [17]:
# 2개의 데이터
logits = torch.tensor([[2.0, 1, 0.1],
                      [0.5, 2.5, 1]])
prob_dim0 = torch.softmax(logits, dim=0)
prob_dim1 = torch.softmax(logits, dim=1)

print('dim=0의 결과')
print(prob_dim0)
print('dim0 열 방향의 합:', prob_dim0.sum(dim=0))

print('dim=1의 결과')
print(prob_dim1)
print('dim1 열 방향의 합:', prob_dim1.sum(dim=1))

dim=0의 결과
tensor([[0.8176, 0.1824, 0.2891],
        [0.1824, 0.8176, 0.7109]])
dim0 열 방향의 합: tensor([1.0000, 1.0000, 1.0000])
dim=1의 결과
tensor([[0.6590, 0.2424, 0.0986],
        [0.0996, 0.7361, 0.1643]])
dim1 열 방향의 합: tensor([1.0000, 1.0000])


#### 4가지 활성함수 비교
| 함수 | 출력범위 | 대표적인 사용위치 |
| --- | --- | ---- |
| Sigmoid | 0 ~ 1 | 이진분류 출력층 |
| ReLU | 0 ~ 양의 무한대 | 일반적인 은닉층 | 
| Tanh | -1 ~ 1 | 일부 은닉층 |
| Softmax | 0 ~ 1 전체합 = 1 | 다중분류 출력층 |

In [21]:
x = torch.tensor([-2., -1, 0, 1, 2])

print('Sigmoid:', torch.sigmoid(x))
print('Relu:', torch.relu(x))
print('Tanh:', torch.tanh(x))

# Softmax 함수는 여러 클래스의 점수에 적용
class_scores = torch.tensor([2, 1, 0.1])
print('Softmax : ', torch.softmax(class_scores, dim=0))

Sigmoid: tensor([0.1192, 0.2689, 0.5000, 0.7311, 0.8808])
Relu: tensor([0., 0., 0., 1., 2.])
Tanh: tensor([-0.9640, -0.7616,  0.0000,  0.7616,  0.9640])
Softmax :  tensor([0.6590, 0.2424, 0.0986])
